# QnA: 

### 1) What is a system prompt?

A system prompt is the instruction given to the model before chat starts. It defines the bot’s role and behavior.

### 2) How does the chatbot remember context?

It stores previous user and assistant messages in a messages list and sends that full list each time.

### 3) How did you restrict the chatbot to one domain?

I wrote a strict system prompt that allows only career-related topics and tells the bot to refuse unrelated questions politely.


# BOT Training and testing

In [25]:
import random
import json
import pandas as pd

In [13]:
#data loading and train,test split

DATA_PATH = "/kaggle/input/datasets/ammarshafiqqasmi/chatbot-training-dataset/chatbot_training_dataset.json"

with open(DATA_PATH, "r", encoding = "utf-8") as f:
    data = json.load(f)

def split_to_df(split_name):
    rows = []
    for item in data["splits"][split_name]:
        user_text = item["messages"][-1]["content"]
        rows.append({
            "id": item["id"],
            "user_text": user_text,
            "assistant_text": item["assistant"],
            "response_type": item["response_type"]
        })
    return pd.DataFrame(rows)

#train, test, and validate split
train_df = split_to_df("train")
val_df = split_to_df("validation")
test_df = split_to_df("test")

print(train_df.head())
print(train_df["response_type"].value_counts())

       id                                          user_text  \
0  tr_001  I am a final-year student. How should I prepar...   
1  tr_002  Can you review this resume bullet: Built API s...   
2  tr_003    What skills are needed for a data analyst role?   
3  tr_004  How do I switch from mechanical engineering to...   
4  tr_005  What should I say when asked Tell me about you...   

                                      assistant_text     response_type  
0  Start with a 6-week plan: 1) data structures a...  in_domain_answer  
1  Make it measurable: Developed REST API service...  in_domain_answer  
2  Core skills: SQL, Excel, statistics, data visu...  in_domain_answer  
3  Use a transition roadmap: 1) statistics and Py...  in_domain_answer  
4  Use Present-Past-Future: who you are now, rele...  in_domain_answer  
response_type
in_domain_answer         15
out_of_domain_refusal     3
Name: count, dtype: int64


In [17]:
# Training and validating using sklearn and logistic regression model

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ("lr", LogisticRegression(max_iter=1000 , class_weight="balanced"))
])

clf.fit(train_df["user_text"], train_df["response_type"])

val_pred = clf.predict(val_df["user_text"])
print(classification_report(val_df["response_type"], val_pred))

                       precision    recall  f1-score   support

     in_domain_answer       1.00      1.00      1.00         1
out_of_domain_refusal       1.00      1.00      1.00         1

             accuracy                           1.00         2
            macro avg       1.00      1.00      1.00         2
         weighted avg       1.00      1.00      1.00         2



In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

in_domain_df = train_df[train_df["response_type"] == "in_domain_answer"].reset_index(drop=True)

retriever_vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_in_domain = retriever_vectorizer.fit_transform(in_domain_df["user_text"])

In [22]:
memory = []  # stores chat turns

REFUSAL = "I focus only on career guidance. Please ask a career-related question."

def chatbot_reply(user_input):
    # 1) domain check
    label = clf.predict([user_input])[0]
    
    if label == "out_of_domain_refusal":
        answer = REFUSAL
    else:
        # 2) retrieval for in-domain answer
        q_vec = retriever_vectorizer.transform([user_input])
        sims = cosine_similarity(q_vec, X_in_domain)[0]
        best_idx = sims.argmax()
        answer = in_domain_df.loc[best_idx, "assistant_text"]

    # 3) save memory
    memory.append({"user": user_input, "bot": answer})
    return answer

In [23]:
tests = [
    "How should I prepare for data analyst interviews?",
    "Can you improve my resume bullet for internship?",
    "Who won the football world cup?"
]

for q in tests:
    print("You:", q)
    print("Bot:", chatbot_reply(q))
    print()

You: How should I prepare for data analyst interviews?
Bot: Start with a 6-week plan: 1) data structures and algorithms, 2) one backend project, 3) system design basics, 4) mock interviews twice a week. Track progress daily.

You: Can you improve my resume bullet for internship?
Bot: Make it measurable: Developed REST API service adopted by 5 internal teams, reducing integration time by 30 percent.

You: Who won the football world cup?
Bot: I focus only on career guidance. Please ask a career-related question.

